# 01 · Data Preparation (Load → EDA → Preprocess → Normalize)

Prudential Life Insurance Assessment. The goal is to predict `Response` (an **ordinal** risk level 1–8); the metric is Quadratic Weighted Kappa.

This notebook only **turns the raw data into a numeric matrix ready for a neural network**, and finally saves it to `../data/processed.npz` for `02_train_eval.ipynb` to load.

Pipeline:
1. Load the data and split it into train / val using `split_indices.csv`
2. EDA: missing values, column types, feature groups
3. Preprocess: drop columns → one-hot → impute + missing flags → standardize (fit on train only)
4. Save the processed data

## Part 1: Load the data and split by the split file

In [27]:
import pandas as pd

pd.set_option("display.max_rows", 60)

# Load the raw training data and the split file.
# Note: this notebook lives in manual/, data is in ../data/
train = pd.read_csv("../data/train.csv")
split = pd.read_csv("../data/split_indices.csv")

print("train:", train.shape)   # (59381, 128) = Id + 126 features + Response
print("split:", split.shape)   # (59381, 2) = Id, split
split["split"].value_counts()

train: (59381, 128)
split: (59381, 2)


split
train    47504
val      11877
Name: count, dtype: int64

In [28]:
# Merge the two tables on Id (same as the last step in data_processing.py).
df = train.merge(split, on="Id")

# Split into two tables by the 'split' column.
train_df = df[df["split"] == "train"].drop(columns="split").reset_index(drop=True)
val_df   = df[df["split"] == "val"].drop(columns="split").reset_index(drop=True)

print("train_df:", train_df.shape)
print("val_df:  ", val_df.shape)

train_df: (47504, 128)
val_df:   (11877, 128)


In [29]:
# Confirm the split is stratified: class ratios should match across train / val.
dist = pd.DataFrame({
    "train": train_df["Response"].value_counts(normalize=True),
    "val":   val_df["Response"].value_counts(normalize=True),
}).sort_index().round(3)
dist

,train,val
Response,,
1,0.105,0.104
2,0.110,0.110
3,0.017,0.017
4,0.024,0.024
5,0.091,0.091
6,0.189,0.189
7,0.135,0.135
8,0.328,0.328


**What we see / how we handle it:**

- After merging, splitting by the `split` column gives **train ≈ 47504 rows, val ≈ 11877 rows** (80/20).
- The `Response` class ratios are almost identical across train and val → this is a **stratified split**, so the validation set faithfully represents the training distribution, and selecting the model by validation Kappa is trustworthy.
- The data is **class-imbalanced**: `Response=8` is ~33%, while `Response=3` is only ~1.7% — a ~20x gap. Keep this in mind when modeling (minority classes are easily ignored).
- All preprocessing statistics (median for imputation, category encoding, mean/variance for standardization) must be **computed on `train_df` only** and then applied to `val_df`, to avoid information leakage.

## Part 2: Exploratory Data Analysis (EDA)

All exploration looks **only at the training set** `train_df`, so we don't peek at validation.

### Q1: Which columns have missing values, and how many?

In [30]:
# Missing ratio per column, sorted descending, top 15.
miss = train_df.isnull().mean().sort_values(ascending=False)
(miss.head(15) * 100).round(1)  # unit: %

Medical_History_10     99.1
Medical_History_32     98.1
Medical_History_24     93.5
Medical_History_15     75.1
Family_Hist_5          70.4
Family_Hist_3          57.6
Family_Hist_2          48.3
Insurance_History_5    42.6
Family_Hist_4          32.3
Employment_Info_6      18.2
Medical_History_1      14.9
Employment_Info_4      11.4
Employment_Info_1       0.0
Medical_Keyword_13      0.0
Medical_Keyword_14      0.0
dtype: float64

In [31]:
# How many columns have any missing values.
print("columns with missing:", (miss > 0).sum())
print("columns missing > 50%:", list(miss[miss > 0.5].index))

columns with missing: 13
columns missing > 50%: ['Medical_History_10', 'Medical_History_32', 'Medical_History_24', 'Medical_History_15', 'Family_Hist_5', 'Family_Hist_3']


**What we see / how we handle it:**

Only **13 columns** have missing values, handled in three tiers:

| Column | Missing % | Decision |
|--------|-----------|----------|
| `Medical_History_10` | ~99% | Almost entirely empty, too little info → **drop** |
| `Medical_History_32` | ~98% | Same → **drop** |
| `Medical_History_24` | ~94% | Same → **drop** |
| `Medical_History_15` | ~75% | Very sparse; could drop, or add a missing flag and impute |
| `Family_Hist_5` | ~70% | Same |
| `Family_Hist_3` | ~58% | **Keep**, median-impute + add missing flag |
| `Family_Hist_2` | ~48% | Same |
| `Insurance_History_5` | ~43% | Same |
| `Family_Hist_4` | ~32% | Same |
| `Employment_Info_6` | ~18% | Median-impute |
| `Medical_History_1` | ~15% | Median-impute |
| `Employment_Info_4` | ~11% | Median-impute |
| `Employment_Info_1` | <0.1% | Median-impute |

Initial plan: **drop the three columns missing >90%**; median-impute the remaining numeric columns using the **training-set median**, and for the high-missing columns add a 0/1 `_isna` flag column (missingness itself may carry signal, e.g. no family history filled in). The imputation median is computed on `train_df` only.

### Q2: Which columns are text, and which are numeric?

In [32]:
# Count columns per dtype.
train_df.dtypes.value_counts()

int64      109
float64     18
str          1
Name: count, dtype: int64

In [33]:
# Find the text (object) columns.
obj_cols = train_df.select_dtypes(include="object").columns
print("text columns:", list(obj_cols))

# Inspect its values.
print("\nProduct_Info_2 values:")
print(sorted(train_df["Product_Info_2"].unique()))

text columns: ['Product_Info_2']

Product_Info_2 values:
['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'B1', 'B2', 'C1', 'C2', 'C3', 'C4', 'D1', 'D2', 'D3', 'D4', 'E1']


/var/folders/mc/hgyk4c3n2lb2r2_kz9ylrhzh0000gn/T/ipykernel_91375/799335420.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = train_df.select_dtypes(include="object").columns


**What we see / how we handle it:**

- Column type breakdown: **~108 int64, 18 float64, 1 object** (`Response` is an integer label, counted in int).
- The only text column is **`Product_Info_2`**, with values like `A1 A2 ... E1` — **19 categories**, a letter+digit combined code.
- Decision: use **one-hot** (19 columns), to avoid treating unordered categories as if they had a numeric magnitude/order.

### Q3: Roughly how many feature groups are there?

In [34]:
# Group features by column-name prefix and count columns per group.
feature_cols = [c for c in train_df.columns if c not in ("Id", "Response")]

prefixes = [
    "Product_Info", "Employment_Info", "InsuredInfo", "Insurance_History",
    "Family_Hist", "Medical_History", "Medical_Keyword",
]
singles = ["Ins_Age", "Ht", "Wt", "BMI"]

groups = {p: sum(c.startswith(p) for c in feature_cols) for p in prefixes}
for s in singles:
    groups[s] = int(s in feature_cols)

group_tbl = pd.Series(groups, name="n_cols").sort_values(ascending=False)
print(group_tbl)
print("\ntotal feature columns:", group_tbl.sum())

Medical_Keyword      48
Medical_History      41
Insurance_History     8
Product_Info          7
InsuredInfo           7
Employment_Info       6
Family_Hist           5
Ins_Age               1
Ht                    1
Wt                    1
BMI                   1
Name: n_cols, dtype: int64

total feature columns: 126


**What we see / data-structure overview (usable directly in the report):**

The 126 features fall into these groups by meaning:

| Group | Columns | Meaning |
|-------|---------|---------|
| `Medical_Keyword_*` | 48 | Whether a medical keyword is present, **0/1 dummy** |
| `Medical_History_*` | 41 | Medical-history related, mostly **ordinal/nominal categories**; missingness concentrates here |
| `Insurance_History_*` | 8 | Insurance history |
| `Product_Info_*` | 7 | Product info (includes the only text column `Product_Info_2`) |
| `InsuredInfo_*` | 7 | Insured-person info |
| `Employment_Info_*` | 6 | Employment info (a few continuous columns with missing values) |
| `Family_Hist_*` | 5 | Family history (continuous, high missing rate) |
| `Ins_Age` / `Ht` / `Wt` / `BMI` | 1 each | Age / height / weight / BMI, **continuous**, already normalized to 0–1 |

Summary: the data is **dominated by medical features** (Keyword 48 + History 41 = 89 columns, ~70%); the 48 Keyword columns are already 0/1 and need no work; the continuous features are mainly `Ins_Age/Ht/Wt/BMI` plus a few `Employment_Info` / `Family_Hist`, which need standardization; missingness is almost entirely concentrated in the `Medical_History` and `Family_Hist` groups.

## Part 3: Preprocessing + Normalization

Four things: (1) drop the 3 columns missing >90%; (2) one-hot `Product_Info_2`; (3) median-impute numeric columns + add `_isna` flags for high-missing ones; (4) standardize.

### The key concept: fit and transform

- **fit**: **compute statistics** from the data — median, mean, variance, and which 19 categories `Product_Info_2` has.
- **transform**: **modify the data using the computed statistics**.

**Rule: fit on the training set only, then transform the training set, the validation set, and test.csv with the same statistics.**
It's like an exam — you can't look at the exam paper before deciding how to study. For example, if `Employment_Info_6`'s median on the training set is 0.25, then missing values in the validation set are also filled with 0.25, not with the validation set's own median.

### Two pitfalls

1. **Columns must align after one-hot**: categories are learned from the training set only, and val/test are generated with the same columns. Use `OneHotEncoder(handle_unknown="ignore")` so unseen or missing categories don't create extra columns.
2. **Add the missing flag before imputation**: first record which cells were originally empty, then fill the median. Here `MissingIndicator` reads the raw data, so it records the blanks before imputation naturally.

In [35]:
# (1) Drop columns with > 90% missing.
drop_cols = ["Medical_History_10", "Medical_History_32", "Medical_History_24"]

# Assign a role to each column.
cat_cols = ["Product_Info_2"]                                   # (2) text column to one-hot encode
all_feats = [c for c in train_df.columns
             if c not in (["Id", "Response"] + drop_cols)]       # drop Id / Response / dropped cols
num_cols = [c for c in all_feats if c not in cat_cols]           # everything else is numeric

# (3) Add an _isna flag for columns missing > 10% (missingness may itself carry signal).
miss_rate = train_df[num_cols].isnull().mean()
flag_cols = list(miss_rate[miss_rate > 0.10].index)

print("drop cols:", drop_cols)
print("one-hot cols:", cat_cols)
print("numeric cols:", len(num_cols))
print("missing-flag cols:", flag_cols)

drop cols: ['Medical_History_10', 'Medical_History_32', 'Medical_History_24']
one-hot cols: ['Product_Info_2']
numeric cols: 122
missing-flag cols: ['Employment_Info_4', 'Employment_Info_6', 'Insurance_History_5', 'Family_Hist_2', 'Family_Hist_3', 'Family_Hist_4', 'Family_Hist_5', 'Medical_History_1', 'Medical_History_15']


In [36]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, MissingIndicator
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# ColumnTransformer runs three branches in parallel, each on its own columns:
#   num  : median impute -> standardize   (all numeric columns)
#   cat  : one-hot                          (Product_Info_2 only)
#   flag : missing indicator                (high-missing columns; reads the raw data,
#          so it records the blanks BEFORE imputation fills them)
pre = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),  # median is learned from train only (at fit)
            ("scale", StandardScaler()),
        ]), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
        ("flag", MissingIndicator(features="all"), flag_cols),
    ],
    remainder="drop",   # Id / Response are in no group -> dropped
)
pre

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_n

In [37]:
# Process test.csv too (no Response, but same feature columns).
test = pd.read_csv("../data/test.csv")

# Fit on train only; val / test are transform-only.
Xtr = pre.fit_transform(train_df)
Xva = pre.transform(val_df)
Xte = pre.transform(test)

# Labels
ytr = train_df["Response"].to_numpy()
yva = val_df["Response"].to_numpy()

# Output column names (num__xxx / cat__Product_Info_2_A1 / flag__xxx)
feat_names = pre.get_feature_names_out()

print("Xtr:", Xtr.shape, " Xva:", Xva.shape, " Xte:", Xte.shape)
print("n features:", len(feat_names))
import collections
kinds = collections.Counter(n.split("__")[0] for n in feat_names)
print("column-type breakdown:", dict(kinds))

Xtr: (47504, 150)  Xva: (11877, 150)  Xte: (19765, 150)
n features: 150
column-type breakdown: {'num': 122, 'cat': 19, 'flag': 9}


In [38]:
# Three sanity checks after preprocessing.
Xtr_df = pd.DataFrame(Xtr, columns=feat_names)

# Check 1: identical column count across train / val / test.
print("same n_cols  train==val:", Xtr.shape[1] == Xva.shape[1],
      " train==test:", Xtr.shape[1] == Xte.shape[1])

# Check 2: no missing values left.
print("total missing:", int(Xtr_df.isnull().sum().sum()))

# Check 3: standardized continuous columns have mean~0, std~1 on train.
print("\nstandardization check (train):")
for c in ["num__Ins_Age", "num__Ht", "num__Wt", "num__BMI"]:
    print(f"  {c:16s} mean={Xtr_df[c].mean(): .2e}  std={Xtr_df[c].std():.3f}")

same n_cols  train==val: True  train==test: True
total missing: 0

standardization check (train):
  num__Ins_Age     mean=-1.46e-16  std=1.000
  num__Ht          mean= 2.87e-16  std=1.000
  num__Wt          mean= 5.44e-17  std=1.000
  num__BMI         mean=-2.50e-16  std=1.000


**Check conclusions:**

- ✅ **Same column count**: train / val / test all become **150 columns** (alignment guaranteed by learning one-hot categories from train only + `handle_unknown="ignore"`).
- ✅ **No missing values**: `isnull().sum().sum() == 0`.
- ✅ **Standardization works**: continuous columns have mean ≈ 0, std ≈ 1 on the training set.

150 columns = 122 numeric columns (standardized) + 19 `Product_Info_2` one-hot columns + 9 `_isna` missing-flag columns.

## Part 4: Save the processed data

Save the processed matrices and labels to a single `.npz` so `02_train_eval.ipynb` can load them directly without re-running preprocessing.

In [39]:
import numpy as np

# Keep test Id for building the submission file later.
test_ids = test["Id"].to_numpy()

np.savez_compressed(
    "../data/processed.npz",
    Xtr=Xtr, ytr=ytr,
    Xva=Xva, yva=yva,
    Xte=Xte, test_ids=test_ids,
    feat_names=np.array(feat_names, dtype=object),
)
print("saved -> ../data/processed.npz")
print("Xtr", Xtr.shape, "| Xva", Xva.shape, "| Xte", Xte.shape)

saved -> ../data/processed.npz
Xtr (47504, 150) | Xva (11877, 150) | Xte (19765, 150)
